
# Multitask learning with Chemprop

Chemprop appeared in most of the top 20, and **every single one of the top five** trained endpoints together rather than separately.

The idea: instead of nine separate models, train one model with a shared
molecular encoder and nine output heads. The encoder has to learn a
representation of a molecule that is useful for all nine tasks at once, and
the endpoints with lots of data end up paying for a better representation that
the data-poor endpoints get to use for free.

---
### Setup

In [ ]:
#@title Getting things all setup...
# Run me first.
%pip -q install rdkit pandas numpy scipy scikit-learn huggingface_hub fsspec lightgbm matplotlib seaborn pyarrow chemprop==2.3.1
!git clone https://github.com/agura-alt/ai4chem_openadmet.git
%cd ai4chem_openadmet


In [ ]:
#@title Imports...
import os, sys
SETUP_DIR = os.path.abspath("Setup")
os.path.isdir(SETUP_DIR) or sys.exit(f"No Setup dir at {SETUP_DIR}; cwd is {os.getcwd()}")

if SETUP_DIR not in sys.path:
    sys.path.insert(0, SETUP_DIR)

assert os.path.exists("Setup/common.py") and os.path.getsize("Setup/common.py") > 1000, (
    "common.py is missing or truncated. Upload it using the folder icon in the "
    "left sidebar, then re-run this cell.")

sys.modules.pop("common", None)            # force a fresh read
import common


Change `your-pair-name` to your team name. It has to match the list of registered teams exactly, and be the same in every notebook &mdash; that is what links your work together.

In [ ]:
# Same folder as every other notebook -- splits, predictions, scores.
common.setup(pair="your-pair-name")


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import os, subprocess
sns.set_style("whitegrid")
WORK = common.workdir()

train = common.load_train()
test  = common.load_test()

display(common.list_splits())

In [ ]:
SPLIT = "random"        # <-- change to whichever you trust
fold, split_meta = common.load_split(train, name=SPLIT)

train_df = train[(fold == "train").to_numpy()].reset_index(drop=True)
val_df   = train[(fold == "val").to_numpy()].reset_index(drop=True)
print(f"{len(train_df)} train / {len(val_df)} val molecules")

### Check your runtime first

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("\n>>> Runtime -> Change runtime type -> GPU, then re-run. <<<")

---
## 1. Loading data -- and a note on sparsity

Your target matrix is mostly `NaN`. Chemprop does not drop those rows &mdash; it
**masks** them. For each molecule, the loss is computed only over the endpoints
that were actually measured, and no gradient flows from the missing ones.

That masking is the entire reason sparse multitask learning works. A molecule
with only an HLM measurement still contributes: it trains the shared encoder,
which improves the representation used by all nine heads. Under nine separate
models, that molecule would have been invisible to eight of them.

In [ ]:
#@title Chemprop helpers...

def write_chemprop_csv(df, path, endpoints):
    """Chemprop reads a CSV of SMILES plus one column per target."""
    cols = ["SMILES"] + list(endpoints)
    out = df[[c for c in cols if c in df.columns]].copy()
    for endpoint in endpoints:
        if endpoint not in out.columns:
            out[endpoint] = np.nan
    out.to_csv(path, index=False)
    return path


def run_chemprop_cmd(cmd):
    """Run a chemprop command and fail readably if the CLI has moved."""
    done = subprocess.run(cmd, capture_output=True, text=True)
    if done.returncode != 0:
        tail = (done.stderr or done.stdout or "").strip().split("\n")[-6:]
        raise RuntimeError(
            "chemprop failed:\n  " + " ".join(cmd[:3]) + " ...\n  "
            + "\n  ".join(tail)
            + "\n\nIf it says 'unrecognized arguments', the CLI changed between\n"
              "versions. Check `!chemprop train --help` and fix the flags below.")
    return done


def run_chemprop(endpoints, tag, training_data, epochs=30, extra_args=()):
    """Train a Chemprop model on `endpoints`. Returns the run directory.

    training_data : the frame to train on -- pass train_df while you are
                    experimenting, and the full `train` when you build the
                    model you actually submit. Nothing is read from globals.
    """
    run_dir = os.path.join(WORK, "chemprop", tag)
    os.makedirs(run_dir, exist_ok=True)

    # Chemprop carves its own small early-stopping slice off what you give it
    # (--split-sizes below), so your validation fold is never seen in training.
    train_path = write_chemprop_csv(training_data,
                                    os.path.join(run_dir, "train.csv"), endpoints)

    cmd = ["chemprop", "train",
           "--data-path", train_path,
           "--split-sizes", "0.9", "0.1", "0.0",   # train / early-stop / unused
           "--task-type", "regression",
           "--smiles-columns", "SMILES",
           "--target-columns", *endpoints,
           "--output-dir", run_dir,
           "--epochs", str(epochs),
           "--num-workers", "0",
           *extra_args]
    print(f"  training {tag} on {len(training_data)} molecules, "
          f"{len(endpoints)} endpoint(s)")
    run_chemprop_cmd(cmd)
    return run_dir


def chemprop_predict(model_dir, target_data, endpoints, tag):
    """Predict `endpoints` for every molecule in target_data."""
    run_dir = os.path.join(model_dir, "pred_" + tag)
    os.makedirs(run_dir, exist_ok=True)
    input_path = write_chemprop_csv(target_data,
                                    os.path.join(run_dir, "input.csv"), endpoints)
    output_path = os.path.join(run_dir, "preds.csv")
    checkpoint = os.path.join(model_dir, "model_0", "best.pt")

    run_chemprop_cmd(["chemprop", "predict",
                      "--test-path", input_path,
                      "--model-path", checkpoint,
                      "--preds-path", output_path,
                      "--smiles-columns", "SMILES"])

    preds = pd.read_csv(output_path)
    out = pd.DataFrame({"Molecule Name": target_data["Molecule Name"].to_numpy()})
    for endpoint in endpoints:
        if endpoint in preds.columns:
            column = endpoint
        else:
            matches = [c for c in preds.columns if endpoint in c]
            if not matches:
                raise KeyError(f"chemprop wrote no column for {endpoint!r}. "
                               f"It produced: {list(preds.columns)}")
            column = matches[0]
        out[endpoint] = preds[column].to_numpy()
    return out

---
## 2. Does sharing actually help?

Take one **data-poor** endpoint and
train it two ways: alone, and grouped with LogD (which has the most data of
any endpoint).

Let's start with `Log_Mouse_BPB` &mdash; few measurements, and physically it is
driven by lipophilicity, which is exactly what LogD measures.

### &#9654;&#65039; Predict first

**Will training Log_Mouse_BPB together with LogD help it, hurt it, or do nothing? And what happens to LogD itself?**

Write your answer here before running the next cell &mdash; one line is enough:

> `your prediction:`

In [ ]:
endpoint1 = "Log_Mouse_BPB"          # the data-poor one

# the data-poor endpoint, alone
dir_solo  = run_chemprop([endpoint1], "solo", train_df, epochs=30)
pred_solo = chemprop_predict(dir_solo, val_df, [endpoint1], "val")
eval_solo = common.score(val_df, pred_solo, f"chemprop-{endpoint1}-solo", SPLIT,
                         endpoints=[endpoint1])

# LogD alone, so you can see what sharing costs the data-RICH endpoint
dir_logd  = run_chemprop(["LogD"], "logd_solo", train_df, epochs=30)
pred_logd = chemprop_predict(dir_logd, val_df, ["LogD"], "val")
eval_logd = common.score(val_df, pred_logd, "chemprop-LogD-solo", SPLIT,
                         endpoints=["LogD"])

# both together: one shared encoder, one head per endpoint
dir_pair  = run_chemprop(["LogD", endpoint1], "with_logd", train_df, epochs=30)
pred_pair = chemprop_predict(dir_pair, val_df, ["LogD", endpoint1], "val")
eval_pair = common.score(val_df, pred_pair, f"chemprop-LogD+{endpoint1}", SPLIT,
                         endpoints=["LogD", endpoint1])

print(f"\n{endpoint1} alone      :", eval_solo.loc[endpoint1, "RAE"].round(3))
print(f"{endpoint1} with LogD  :", eval_pair.loc[endpoint1, "RAE"].round(3))
print("LogD alone                :", eval_logd.loc["LogD", "RAE"].round(3))
print(f"LogD with {endpoint1}:", eval_pair.loc["LogD", "RAE"].round(3))

  training solo on 4261 molecules, 1 endpoint(s)


/content/ai4chem_openadmet/Setup/common.py:842: UserWarning: 'chemprop-Log_Mouse_BPB-solo' was scored on 1 of 9 endpoints. Its MA-RAE averages over those only, so it is not comparable with a full model and will NOT count toward the MA-RAE leaderboard. Useful for the per-endpoint boards, and for comparing against another partial model on the same endpoints.
  warnings.warn(


logged chemprop-Log_Mouse_BPB-solo on 'random': MA-RAE = 0.472
  training logd_solo on 4261 molecules, 1 endpoint(s)


/content/ai4chem_openadmet/Setup/common.py:842: UserWarning: 'chemprop-LogD-solo' was scored on 1 of 9 endpoints. Its MA-RAE averages over those only, so it is not comparable with a full model and will NOT count toward the MA-RAE leaderboard. Useful for the per-endpoint boards, and for comparing against another partial model on the same endpoints.
  warnings.warn(


logged chemprop-LogD-solo on 'random': MA-RAE = 0.332
  training with_logd on 4261 molecules, 2 endpoint(s)
logged chemprop-LogD+Log_Mouse_BPB on 'random': MA-RAE = 0.410

Log_Mouse_BPB alone      : 0.472
Log_Mouse_BPB with LogD  : 0.465
LogD alone                : 0.332
LogD with Log_Mouse_BPB: 0.355


/content/ai4chem_openadmet/Setup/common.py:842: UserWarning: 'chemprop-LogD+Log_Mouse_BPB' was scored on 2 of 9 endpoints. Its MA-RAE averages over those only, so it is not comparable with a full model and will NOT count toward the MA-RAE leaderboard. Useful for the per-endpoint boards, and for comparing against another partial model on the same endpoints.
  warnings.warn(


Did adding the endpoints add to the individual scores? Did helping the small endpoint *cost* the large
one anything?

Sharing is not free, and when tasks conflict, the shared encoder
has to compromise. Chemprop implements a specific kind of multi-task learning known as hard parameter-sharing -- it's not the only way to do it, and you should try other approaches!

---
## 3. Which endpoints should be grouped?

You cannot test every option. The number of ways to partition nine endpoints
into groups is 21,147!

Here's some information to help you test specific groupings:

1. **The correlation heatmap.** Is label correlation the same thing as task affinity?
2. **Chemistry.** Lipophilicity drives permeability and protein binding. The
   two clearance assays are the same experiment in two species. The two Caco-2 readouts come off the same plate.
3. **What worked before.** Task-affinity grouping has a literature, and
   the top finishers used it.

In [ ]:
corr = train[common.ENDPOINTS].corr(method="spearman", min_periods=30)
overlap = train[common.ENDPOINTS].notna().astype(int).T @ train[common.ENDPOINTS].notna().astype(int)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, ax=axes[0], cbar=False)
axes[0].set_title("Spearman correlation (co-measured molecules only)")
sns.heatmap(overlap, annot=True, fmt="d", cmap="Greys", ax=axes[1], cbar=False)
axes[1].set_title("How many molecules have BOTH endpoints")
plt.tight_layout(); plt.show()

**Read the two panels together.** Which correlations would you actually
trust, given how many molecules they were computed on?

One trap: Two endpoints can
help each other because they need the same internal representation of a
molecule, even when their measured values are uncorrelated.

In [ ]:
def evaluate_grouping(name, groups, training_data, target_data, epochs=30):
    """Train one model per group, stitch the predictions back together, log it."""
    parts = []
    for i, group in enumerate(groups):
        run_dir = run_chemprop(group, f"{name}_{i}", training_data, epochs=epochs)
        parts.append(chemprop_predict(run_dir, target_data, group, "val")
                     .set_index("Molecule Name"))
    merged = pd.concat(parts, axis=1).reset_index()

    covered = [e for group in groups for e in group]
    eval = common.score(target_data, merged, f"chemprop-{name}", SPLIT,
                        endpoints=covered, note=f"grouping={name}")
    print(f"\n{name}: MA-RAE = {eval['RAE'].mean():.3f}")
    return eval, merged

In [ ]:
# Try one or two more. Each grouping costs one training run per group!
grouping = [common.ENDPOINTS]          # everything in a single model
eval_all, pred_all = evaluate_grouping("all_together", grouping, train_df, val_df)

eval_all.round(3)

In [ ]:
# your group -- fill these in with whatever partition you want to test
grouping = [["LogD", "Log_Mouse_PPB"],
            [..., ...],
            [..., ..., ...]]

eval_mine, pred_mine = evaluate_grouping("my_grouping", grouping, train_df, val_df)
eval_mine.round(3)

---
## Other things to try

- What Chemprop does is **hard sharing**: one encoder, shared by every task, with
separate output heads. Every task sees the same molecular representation. **Soft sharing** gives each task its own encoder and adds a penalty encouraging
them to stay similar &mdash; more flexible when tasks partly conflict, but it will require custom implementation!

- You can also pass chemical descriptors to the Chemprop model in addition to the learned fingerprints.
To pass molecule level descriptors (rdkit descriptors, morgan fingerprints, mordred, etc.),
try `--descriptors-path`
To pass atom or bond-level descriptors, try --atom-features-path, --bond-features-path, or --atom-descriptors-path, --bond-descriptors-path.

- You can also try the opposite: taking the Chemprop encoding as input features for a different model type, like random forest -- perhaps an individual random forest for each endpoint. This is a type of soft parameter sharing; training an encoding with all descriptors but having a modeling component that is not shared.

- Try using pretrained weights! GNNs are often data-hungry, and Chemprop comes with pretrained models that could help with that (see Chemeleon in `Pretrained.ipynb`)

There's lots to play around with Chemprop, go crazy!

---
## Save your work

Give it a name you will recognise! `Ensembles` can combine this
with anything else you have made today.

### Make a submission
Go back and try some other splits if you like with your best grouping! Then, make a submission.
Note: if you used `evaluate_grouping`, your model name is `chemprop-'whatever name you passed to evaluate_grouping'`


In [ ]:
common.score_matrix()

** Note: ** The below cell saves endpoint predictions in a way that assumes the grouping only contains one of each endpoint. There are valid reasons you might want to do repeat endpoints across groupings, but you'll have to write your own code to save the predictions for submission in that case!

You'll have to write a custom prediction saver in any case where you might want to use different models for different endpoints -- and you should probably repeat that process when you call common.score as well.

In [ ]:
# The model you submit trains on EVERY labelled molecule -- pass `train`
# rather than train_df. The split above was only for estimating the score.

BEST_GROUPING = ...     # <-- the grouping that won
MODEL = ...        # <-- must match what you logged above
SPLIT = ...

preds = []
for i, group in enumerate(BEST_GROUPING):
    run_dir = run_chemprop(group, f"final_{i}", train, epochs=40)
    preds.append(chemprop_predict(run_dir, test, group, "test")
                 .set_index("Molecule Name"))

test_preds = pd.concat(preds, axis=1).reset_index()
test_preds.head()

In [ ]:
common.prepare_submission(test_preds, MODEL, SPLIT,
                          why=f"Chemprop v2 multitask, grouping={BEST_GROUPING}")